In [9]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

# =============================================================================
# CONFIGURATION
# =============================================================================
IMAGE_PATH = "./input/bg1.jpg"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = (400, 400, 3)   # H x W x C
FIG_DPI  = 150


# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def save_fig(img_bgr, filename, title=""):
    """Luu hinh (BGR) thanh PNG."""
    path = os.path.join(OUTPUT_DIR, filename)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=FIG_DPI)
    ax.imshow(img_rgb)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    print(f"[SAVED] {path}")
    return path


def draw_grid_overlay(img_bgr, step=40):
    """Ve luoi len anh (in-place copy)."""
    out = img_bgr.copy()
    H, W = out.shape[:2]
    for x in range(0, W, step):
        cv2.line(out, (x,0), (x,H), (0,0,200), 1)
    for y in range(0, H, step):
        cv2.line(out, (0,y), (W,y), (0,0,200), 1)
    return out


def apply_transform_manual(img, M, out_size=None):
    """
    Ap dung bien doi affine/projective bang cach tu nhan ma tran,
    thay vi goi warpAffine/warpPerspective truc tiep.
    Chi dung de MINH HOA gia tri pixel tai (x,y) ung voi toa do anh goc.
    Thuc te: vong lap Python rat cham nen chi ap dung tren anh nho.
    """
    H, W = img.shape[:2]
    if out_size is None:
        out_size = (W, H)
    OW, OH = out_size
    out = np.zeros((OH, OW, 3), dtype=np.uint8)

    is_projective = (M.shape == (3, 3))
    if not is_projective:
        # Dua affine 2x3 len 3x3
        M3 = np.vstack([M, [0, 0, 1]])
    else:
        M3 = M.copy()

    M_inv = np.linalg.inv(M3)

    for v in range(OH):
        for u in range(OW):
            p_dst = np.array([u, v, 1.0])
            p_src = M_inv @ p_dst
            if is_projective:
                p_src /= p_src[2]
            xs, ys = int(round(p_src[0])), int(round(p_src[1]))
            if 0 <= xs < W and 0 <= ys < H:
                out[v, u] = img[ys, xs]
    return out




In [10]:
# =============================================================================
# A. TRANSLATION
# =============================================================================

def build_translation_matrix(tx, ty):
    """
    Ma tran dich chuyen trong toa do thuan nhat:
    [1 0 tx]
    [0 1 ty]
    [0 0  1]
    warpAffine nhan dang affine 2x3.
    """
    M = np.float32([[1, 0, tx],
                    [0, 1, ty]])
    return M


def demo_translation(img, tx=80, ty=50):
    H, W = img.shape[:2]
    M = build_translation_matrix(tx, ty)
    result = cv2.warpAffine(img, M, (W, H))
    return result, M


In [11]:
# =============================================================================
# B. ROTATION
# =============================================================================

def build_rotation_matrix(cx, cy, angle_deg, scale=1.0):
    """
    Xay dung ma tran xoay quanh tam (cx, cy) voi goc theta (do).
    R = T(cx,cy) * Rot(theta) * T(-cx,-cy)
    Dang affine 2x3:
    [cos(t)*s  -sin(t)*s  (1-s*cos)*cx + s*sin*cy]
    [sin(t)*s   cos(t)*s  -s*sin*cx + (1-s*cos)*cy]
    """
    theta = np.deg2rad(angle_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    s = scale
    M = np.float32([
        [ s*cos_t, -s*sin_t, (1 - s*cos_t)*cx + s*sin_t*cy],
        [ s*sin_t,  s*cos_t, -s*sin_t*cx + (1 - s*cos_t)*cy]
    ])
    return M


def demo_rotation(img, angle_deg=35):
    H, W = img.shape[:2]
    cx, cy = W / 2, H / 2
    M = build_rotation_matrix(cx, cy, angle_deg)
    result = cv2.warpAffine(img, M, (W, H))
    return result, M

In [12]:
# =============================================================================
# C. SCALING
# =============================================================================

def build_scaling_matrix(sx, sy, cx=0, cy=0):
    """
    Ma tran ti le quanh tam (cx, cy):
    [sx 0  cx*(1-sx)]
    [0  sy cy*(1-sy)]
    """
    M = np.float32([
        [sx, 0,  cx*(1 - sx)],
        [0,  sy, cy*(1 - sy)]
    ])
    return M


def demo_scaling(img, sx=0.6, sy=0.6):
    H, W = img.shape[:2]
    cx, cy = W / 2, H / 2
    M = build_scaling_matrix(sx, sy, cx, cy)
    result = cv2.warpAffine(img, M, (W, H))
    return result, M

In [13]:
# =============================================================================
# VISUALIZATION: ALL TRANSFORMATIONS
# =============================================================================

def visualize_all(original, results_dict):
    """Tong hop tat ca phep bien doi trong 1 figure."""
    titles = list(results_dict.keys())
    imgs   = list(results_dict.values())

    n = len(imgs) + 1
    cols = 3
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*5), dpi=FIG_DPI)
    axes = axes.flatten()

    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    for i, (title, img) in enumerate(zip(titles, imgs), start=1):
        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title, fontsize=12, fontweight='bold')
        axes[i].axis('off')

    for j in range(i+1, len(axes)):
        axes[j].axis('off')

    fig.suptitle('Geometric Transformations – Computer Vision Demo',
                 fontsize=14, fontweight='bold', y=1.01)
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'all_transforms.png')
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    print(f"[SAVED] {path}")

# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():
    print("=" * 60)
    print("  Geometric Transformations – Computer Vision Demo")
    print("=" * 60)

    original = cv2.imread(IMAGE_PATH)
    if original is None:
        raise FileNotFoundError(f"Khong doc duoc anh: '{IMAGE_PATH}'. Kiem tra lai duong dan.")
    original = cv2.resize(original, (400, 400))   # chuan hoa kich thuoc
    save_fig(original, 'original.png', 'Original Image')

    # 2. Translation
    trans_img, M_trans = demo_translation(original, tx=80, ty=50)
    save_fig(trans_img, 'translation.png', f'Translation (tx=80, ty=50)')
    print(f"\nTranslation Matrix (2x3):\n{M_trans}")

    # 3. Rotation
    rot_img, M_rot = demo_rotation(original, angle_deg=35)
    save_fig(rot_img, 'rotation.png', 'Rotation (35°, center)')
    print(f"\nRotation Matrix (2x3):\n{M_rot}")

    # 4. Scaling
    scale_img, M_scale = demo_scaling(original, sx=0.6, sy=0.6)
    save_fig(scale_img, 'scaling.png', 'Scaling (sx=sy=0.6)')
    print(f"\nScaling Matrix (2x3):\n{M_scale}")

    # 9. Tong hop hinh
    results = {
        'Translation':  trans_img,
        'Rotation':     rot_img,
        'Scaling':      scale_img,
    }
    visualize_all(original, results)
    
    # 10. In bang tom tat
    print("\n" + "=" * 60)
    print("  Summary of Transformation Matrices")
    print("=" * 60)
    print(f"Translation : 2x3 affine, 2 DoF")
    print(f"Rotation    : 2x3 affine, 3 DoF (+ scale = similarity)")
    print(f"Scaling     : 2x3 affine, 2 DoF (anisotropic)")
    print("\n[DONE] All figures saved to ./figures/")

if __name__ == "__main__":
    main()

  Geometric Transformations – Computer Vision Demo
[SAVED] output\original.png
[SAVED] output\translation.png

Translation Matrix (2x3):
[[ 1.  0. 80.]
 [ 0.  1. 50.]]
[SAVED] output\rotation.png

Rotation Matrix (2x3):
[[  0.81915206  -0.57357645 150.88487   ]
 [  0.57357645   0.81915206 -78.54569   ]]
[SAVED] output\scaling.png

Scaling Matrix (2x3):
[[ 0.6  0.  80. ]
 [ 0.   0.6 80. ]]
[SAVED] output\all_transforms.png

  Summary of Transformation Matrices
Translation : 2x3 affine, 2 DoF
Rotation    : 2x3 affine, 3 DoF (+ scale = similarity)
Scaling     : 2x3 affine, 2 DoF (anisotropic)

[DONE] All figures saved to ./figures/
